In [ ]:
!pip install nba_api


In [ ]:
from nba_api.stats.endpoints import leaguedashplayerstats
import pandas as pd
import time

In [ ]:
all_data = []

for season in seasons:
    print(f"{season} çekiliyor...")
    stats = leaguedashplayerstats.LeagueDashPlayerStats(
        season=season,
        per_mode_detailed="PerGame"
    )
    df = stats.get_data_frames()[0]
    df["SEASON"] = season
    all_data.append(df)
    time.sleep(2)

nba_df = pd.concat(all_data, ignore_index=True)
print(f"Toplam satır: {len(nba_df)}")
print(nba_df.head())

2010-11 çekiliyor...
2011-12 çekiliyor...
2012-13 çekiliyor...
2013-14 çekiliyor...
2014-15 çekiliyor...
2015-16 çekiliyor...
2016-17 çekiliyor...
2017-18 çekiliyor...
2018-19 çekiliyor...
2019-20 çekiliyor...
2020-21 çekiliyor...
2021-22 çekiliyor...
2022-23 çekiliyor...
2023-24 çekiliyor...
Toplam satır: 7190
   PLAYER_ID    PLAYER_NAME NICKNAME     TEAM_ID TEAM_ABBREVIATION   AGE  GP  \
0     201985       AJ Price       AJ  1610612754               IND  24.0  50   
1     201166   Aaron Brooks    Aaron  1610612756               PHX  26.0  59   
2     201189     Aaron Gray    Aaron  1610612740               NOH  26.0  41   
3     201151       Acie Law     Acie  1610612744               GSW  26.0  51   
4       1733  Al Harrington       Al  1610612743               DEN  31.0  73   

    W   L  W_PCT  ...  PF_RANK  PFD_RANK  PTS_RANK  PLUS_MINUS_RANK  \
0  22  28  0.440  ...      351       198       240              161   
1  26  33  0.441  ...      205       131       132              

In [ ]:
columns_needed = [
    "PLAYER_ID", "PLAYER_NAME", "TEAM_ABBREVIATION", "AGE", "GP",
    "MIN", "PTS", "AST", "TOV", "REB",
    "STL", "BLK", "FG3_PCT", "FT_PCT",
    "FGA", "FTA", "FGM",
    "PLUS_MINUS", "SEASON"
]

nba_clean = nba_df[columns_needed].copy()

# True Shooting % manuel hesapla: PTS / (2 * (FGA + 0.44 * FTA))
nba_clean["TS_PCT"] = nba_clean["PTS"] / (2 * (nba_clean["FGA"] + 0.44 * nba_clean["FTA"]))

# Minimum 1000 dakika filtresi
nba_clean = nba_clean[nba_clean["MIN"] * nba_clean["GP"] >= 1000]

print(f"Filtrelenmiş satır sayısı: {len(nba_clean)}")
print(nba_clean.head())

Filtrelenmiş satır sayısı: 3683
   PLAYER_ID    PLAYER_NAME TEAM_ABBREVIATION   AGE  GP   MIN   PTS  AST  TOV  \
1     201166   Aaron Brooks               PHX  26.0  59  21.8  10.7  3.9  1.7   
4       1733  Al Harrington               DEN  31.0  73  22.8  10.5  1.4  1.5   
5     201143     Al Horford               ATL  25.0  77  35.1  15.3  3.5  1.5   
6       2744   Al Jefferson               UTA  26.0  82  35.9  18.6  1.8  1.3   
7     201154    Al Thornton               GSW  27.0  71  19.5   7.4  0.9  0.9   

   REB  STL  BLK  FG3_PCT  FT_PCT   FGA  FTA  FGM  PLUS_MINUS   SEASON  \
1  1.3  0.6  0.1    0.297   0.886   9.9  2.4  3.7        -2.8  2010-11   
4  4.5  0.5  0.1    0.357   0.735   9.2  1.6  3.8         1.8  2010-11   
5  9.3  0.8  1.0    0.500   0.798  12.0  2.4  6.7         0.6  2010-11   
6  9.7  0.6  1.9    0.000   0.761  16.1  3.5  8.0        -2.0  2010-11   
7  3.0  0.5  0.2    0.154   0.775   6.1  2.0  2.9        -2.0  2010-11   

     TS_PCT  
1  0.488317  
4  0.530

In [ ]:
nba_clean.to_csv("nba_data.csv", index=False)
print("Kaydedildi!")

Kaydedildi!


In [ ]:
from nba_api.stats.static import players
from nba_api.stats.endpoints import commonplayerinfo

player_ids = nba_clean["PLAYER_ID"].unique()
print(f"Toplam unique oyuncu: {len(player_ids)}")

ModuleNotFoundError: No module named 'nba_api'

In [ ]:
from nba_api.stats.endpoints import commonallplayers

all_players = commonallplayers.CommonAllPlayers(
    is_only_current_season=0,
    league_id="00",
    season="2023-24"
)

players_df = all_players.get_data_frames()[0]
print(players_df.columns.tolist())
print(players_df.head())

['PERSON_ID', 'DISPLAY_LAST_COMMA_FIRST', 'DISPLAY_FIRST_LAST', 'ROSTERSTATUS', 'FROM_YEAR', 'TO_YEAR', 'PLAYERCODE', 'PLAYER_SLUG', 'TEAM_ID', 'TEAM_CITY', 'TEAM_NAME', 'TEAM_ABBREVIATION', 'TEAM_CODE', 'TEAM_SLUG', 'GAMES_PLAYED_FLAG', 'OTHERLEAGUE_EXPERIENCE_CH']
   PERSON_ID DISPLAY_LAST_COMMA_FIRST   DISPLAY_FIRST_LAST  ROSTERSTATUS  \
0      76001          Abdelnaby, Alaa       Alaa Abdelnaby             0   
1      76002         Abdul-Aziz, Zaid      Zaid Abdul-Aziz             0   
2      76003     Abdul-Jabbar, Kareem  Kareem Abdul-Jabbar             0   
3         51      Abdul-Rauf, Mahmoud   Mahmoud Abdul-Rauf             0   
4       1505       Abdul-Wahad, Tariq    Tariq Abdul-Wahad             0   

  FROM_YEAR TO_YEAR                   PLAYERCODE          PLAYER_SLUG  \
0      1990    1994       HISTADD_alaa_abdelnaby       alaa_abdelnaby   
1      1968    1977      HISTADD_zaid_abdul-aziz      zaid_abdul-aziz   
2      1969    1988  HISTADD_kareem_abdul-jabbar  kareem_

In [ ]:
from nba_api.stats.endpoints import leaguedashplayerstats
from nba_api.stats.endpoints import playerindex

player_idx = playerindex.PlayerIndex(season="2023-24")
print(player_idx.get_data_frames()[0].columns.tolist())

['PERSON_ID', 'PLAYER_LAST_NAME', 'PLAYER_FIRST_NAME', 'PLAYER_SLUG', 'TEAM_ID', 'TEAM_SLUG', 'IS_DEFUNCT', 'TEAM_CITY', 'TEAM_NAME', 'TEAM_ABBREVIATION', 'JERSEY_NUMBER', 'POSITION', 'HEIGHT', 'WEIGHT', 'COLLEGE', 'COUNTRY', 'DRAFT_YEAR', 'DRAFT_ROUND', 'DRAFT_NUMBER', 'ROSTER_STATUS', 'FROM_YEAR', 'TO_YEAR', 'PTS', 'REB', 'AST', 'STATS_TIMEFRAME']


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving archive.zip to archive.zip


In [ ]:
import pandas as pd
kaggle_df = pd.read_csv("archive.zip")
print(kaggle_df.columns.tolist())
print(kaggle_df.head())

['normalized_name', 'age', 'player_height', 'player_weight', 'college', 'country', 'draft_year', 'draft_round', 'draft_number', 'pts', 'reb', 'ast', 'season', 'Pos.x', 'MP.x', 'G.x', 'eFG.', 'X3P', 'X3PA', 'X3P.', 'X3PAr', 'X2P', 'X2PA', 'X2P.', 'FT', 'FTA', 'FT.', 'PER', 'TS.', 'TRB.', 'AST.', 'TOV.', 'USG.', 'WS', 'VORP', 'BPM']
    normalized_name  age  player_height  player_weight          college  \
0     Allen Iverson   26         183.58          74.61       Georgetown   
1  Jerry Stackhouse   26         198.15          99.23   North Carolina   
2  Shaquille O'Neal   29         216.31         142.56  Louisiana State   
3       Kobe Bryant   22         201.60          95.20              NaN   
4      Vince Carter   24         198.64         102.34   North Carolina   

  country draft_year draft_round draft_number   pts  ...    FT.   PER    TS.  \
0     USA       1996           1            1  31.1  ...  0.814  24.0  0.518   
1     USA       1995           1            3  29.8  ...

In [ ]:
print(kaggle_df["Pos.x"].value_counts())
print(f"\nKaggle sezon formatı: {kaggle_df['season'].unique()[:5]}")

Pos.x
SG    2815
PF    2747
C     2724
PG    2592
SF    2513
Name: count, dtype: int64

Kaggle sezon formatı: ['2000-01' '2001-02' '2002-03' '2003-04' '2004-05']


In [ ]:
print(kaggle_df["Pos.x"].value_counts())
print(kaggle_df["season"].unique())

Pos.x
SG    2815
PF    2747
C     2724
PG    2592
SF    2513
Name: count, dtype: int64
['2000-01' '2001-02' '2002-03' '2003-04' '2004-05' '2005-06' '2006-07'
 '2007-08' '2008-09' '2009-10' '2010-11' '2011-12' '2012-13' '2013-14'
 '2014-15' '2015-16' '2016-17' '2017-18' '2018-19' '2019-20' '2020-21'
 '2021-22' '2022-23' '2023-24' '1996-97' '1997-98' '1998-99' '1999-00']


In [ ]:
print(nba_final.columns.tolist())

['PLAYER_ID', 'PLAYER_NAME', 'TEAM_ABBREVIATION', 'AGE', 'GP', 'MIN', 'PTS', 'AST', 'TOV', 'REB', 'STL', 'BLK', 'FG3_PCT', 'FT_PCT', 'FGA', 'FTA', 'FGM', 'PLUS_MINUS', 'SEASON', 'TS_PCT', 'POSITION_x', 'POSITION_y', 'TS_PCT_KAG', 'PER', 'BPM', 'VORP', 'WS']


In [ ]:
# kaggle'dan gelen temiz pozisyon
nba_final = nba_final.drop(columns=["POSITION_x"])
nba_final = nba_final.rename(columns={"POSITION_y": "POSITION"})

print(nba_final["POSITION"].value_counts())
print(f"Toplam satır: {len(nba_final)}")

POSITION
SG    829
PG    738
PF    725
SF    668
C     610
Name: count, dtype: int64
Toplam satır: 3570


In [ ]:
nba_final.to_csv("nba_final.csv", index=False)

from google.colab import files
files.download("nba_final.csv")

print("Dataset hazır!")
print(f"Satır sayısı: {len(nba_final)}")
print(f"Sütun sayısı: {len(nba_final.columns)}")
print(nba_final.dtypes)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Dataset hazır!
Satır sayısı: 3570
Sütun sayısı: 26
PLAYER_ID              int64
PLAYER_NAME           object
TEAM_ABBREVIATION     object
AGE                  float64
GP                     int64
MIN                  float64
PTS                  float64
AST                  float64
TOV                  float64
REB                  float64
STL                  float64
BLK                  float64
FG3_PCT              float64
FT_PCT               float64
FGA                  float64
FTA                  float64
FGM                  float64
PLUS_MINUS           float64
SEASON                object
TS_PCT               float64
POSITION              object
TS_PCT_KAG           float64
PER                  float64
BPM                  float64
VORP                 float64
WS                   float64
dtype: object
